# Qwen3 Music Chatbot LoRA Fine-tuning on Google Colab

This notebook fine-tunes Qwen3-0.6B using LoRA for music knowledge graph chatbot.

**Features:**
- Parameter-efficient fine-tuning with LoRA
- 4-bit quantization for memory efficiency
- Optimized for Google Colab free tier
- Gradient checkpointing
- Early stopping and checkpointing

**Requirements:**
- Upload your training data: `music_qa_train.json` and `music_qa_val.json`
- Enable GPU runtime (Runtime → Change runtime type → GPU)

In [4]:
!nvidia-smi


Mon Dec  1 04:54:32 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Install required packages
!pip install torch transformers peft trl datasets accelerate bitsandbytes wandb

In [6]:
# Import libraries
import os
import sys
import json
import logging
from pathlib import Path
from typing import Optional, Dict, Any
from dataclasses import dataclass, field

import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
from trl import SFTTrainer, SFTConfig

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Default configurations
DEFAULT_MODEL_NAME = "Qwen/Qwen3-0.6B"
INSTRUCT_MODEL_NAME = "Qwen/Qwen3-0.6B"

In [7]:
# Configuration dataclasses
@dataclass
class ModelArguments:
    """Arguments for model configuration."""
    model_name: str = field(
        default=INSTRUCT_MODEL_NAME,
        metadata={"help": "HuggingFace model name or path"}
    )
    use_4bit: bool = field(
        default=True,
        metadata={"help": "Use 4-bit quantization"}
    )
    use_8bit: bool = field(
        default=False,
        metadata={"help": "Use 8-bit quantization"}
    )
    trust_remote_code: bool = field(
        default=True,
        metadata={"help": "Trust remote code from HuggingFace"}
    )


@dataclass
class LoraArguments:
    """Arguments for LoRA configuration."""
    lora_r: int = field(
        default=16,
        metadata={"help": "LoRA rank"}
    )
    lora_alpha: int = field(
        default=32,
        metadata={"help": "LoRA alpha scaling factor"}
    )
    lora_dropout: float = field(
        default=0.05,
        metadata={"help": "LoRA dropout rate"}
    )
    target_modules: str = field(
        default="q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",
        metadata={"help": "Target modules for LoRA (comma-separated)"}
    )


@dataclass
class DataArguments:
    """Arguments for data configuration."""
    train_data: str = field(
        default="data/training/music_qa_train.json",
        metadata={"help": "Path to training data"}
    )
    val_data: str = field(
        default="data/training/music_qa_val.json",
        metadata={"help": "Path to validation data"}
    )
    max_seq_length: int = field(
        default=1024,
        metadata={"help": "Maximum sequence length"}
    )

In [8]:
# Data loading function
def load_training_data(train_path: str, val_path: str) -> tuple:
    """Load training and validation datasets."""
    logger.info(f"Loading training data from {train_path}")
    
    with open(train_path, 'r', encoding='utf-8') as f:
        train_data = json.load(f)
    
    with open(val_path, 'r', encoding='utf-8') as f:
        val_data = json.load(f)
    
    # Convert to Dataset format
    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)
    
    logger.info(f"Loaded {len(train_dataset)} training examples")
    logger.info(f"Loaded {len(val_dataset)} validation examples")
    
    return train_dataset, val_dataset

In [9]:
# Model and quantization setup
def create_quantization_config(use_4bit: bool = True, use_8bit: bool = False):
    """Create quantization configuration."""
    if use_4bit:
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
    elif use_8bit:
        return BitsAndBytesConfig(
            load_in_8bit=True
        )
    return None


def create_lora_config(
    r: int = 16,
    alpha: int = 32,
    dropout: float = 0.05,
    target_modules: list = None
) -> LoraConfig:
    """Create LoRA configuration."""
    if target_modules is None:
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ]
    
    return LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=target_modules,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )


def load_model_and_tokenizer(
    model_name: str,
    quantization_config: Optional[BitsAndBytesConfig] = None,
    trust_remote_code: bool = True
) -> tuple:
    """Load model and tokenizer."""
    logger.info(f"Loading model: {model_name}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=trust_remote_code
    )
    
    # Set padding token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    # Model loading arguments
    model_kwargs = {
        "trust_remote_code": trust_remote_code,
        "device_map": "auto",
    }
    
    if quantization_config:
        model_kwargs["quantization_config"] = quantization_config
    else:
        model_kwargs["torch_dtype"] = torch.float16
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        **model_kwargs
    )
    
    # Prepare for k-bit training if quantized
    if quantization_config:
        model = prepare_model_for_kbit_training(model)
    
    logger.info(f"Model loaded on device: {model.device}")
    
    return model, tokenizer

In [ ]:
# Data formatting function - FIXED: Use tokenizer's chat template for consistency
_tokenizer_for_formatting = None

def get_formatting_tokenizer():
    """Get tokenizer for formatting (lazy load)."""
    global _tokenizer_for_formatting
    if _tokenizer_for_formatting is None:
        _tokenizer_for_formatting = AutoTokenizer.from_pretrained(
            INSTRUCT_MODEL_NAME,
            trust_remote_code=True
        )
    return _tokenizer_for_formatting

def formatting_func(example: Dict) -> str:
    """Format example for training using tokenizer's native chat template.
    
    This ensures consistency between training format and inference format.
    """
    messages = example.get('messages', [])
    tokenizer = get_formatting_tokenizer()
    
    # Use tokenizer's native chat template for consistency
    # add_generation_prompt=False because we want to include the assistant's response in training
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False
    )
    
    return text

## Upload Training Data

Upload your `music_qa_train.json` and `music_qa_val.json` files, or modify the paths below to point to your data location.

In [11]:
# Upload data files (run this cell and select your files)
from google.colab import files

print("Please upload your training data files:")
print("- music_qa_train.json")
print("- music_qa_val.json")

uploaded = files.upload()

# Create data directory
os.makedirs('data/training', exist_ok=True)

# Move uploaded files to correct location
for filename in uploaded.keys():
    if 'train' in filename:
        os.rename(filename, 'data/training/music_qa_train.json')
    elif 'val' in filename:
        os.rename(filename, 'data/training/music_qa_val.json')

print("Data files uploaded successfully!")

Please upload your training data files:
- music_qa_train.json
- music_qa_val.json


KeyboardInterrupt: 

In [ ]:
# Main training function (optimized for Colab)
def train(
    model_args: ModelArguments,
    lora_args: LoraArguments,
    data_args: DataArguments,
    output_dir: str = "models/qwen3-music-lora",
    num_epochs: int = 2,  # Reduced for Colab
    batch_size: int = 1,  # Smaller batch size for Colab
    gradient_accumulation_steps: int = 8,  # Increased accumulation
    learning_rate: float = 2e-4,
    warmup_ratio: float = 0.1,
    logging_steps: int = 5,  # More frequent logging
    save_steps: int = 50,  # Save more frequently
    eval_steps: int = 50,  # Evaluate more frequently
    max_grad_norm: float = 0.3,
    use_wandb: bool = False,
    run_name: str = "qwen3-music-lora-colab"
):
    """Main training function optimized for Colab."""
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load data
    train_dataset, val_dataset = load_training_data(
        data_args.train_data,
        data_args.val_data
    )
    
    # Create quantization config
    quant_config = create_quantization_config(
        use_4bit=model_args.use_4bit,
        use_8bit=model_args.use_8bit
    )
    
    # Load model and tokenizer
    model, tokenizer = load_model_and_tokenizer(
        model_args.model_name,
        quantization_config=quant_config,
        trust_remote_code=model_args.trust_remote_code
    )
    
    # Create LoRA config
    target_modules = lora_args.target_modules.split(',') if isinstance(lora_args.target_modules, str) else lora_args.target_modules
    lora_config = create_lora_config(
        r=lora_args.lora_r,
        alpha=lora_args.lora_alpha,
        dropout=lora_args.lora_dropout,
        target_modules=target_modules
    )
    
    # Apply LoRA
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # Training arguments optimized for Colab
    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        warmup_ratio=warmup_ratio,
        logging_steps=logging_steps,
        save_steps=save_steps,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_grad_norm=max_grad_norm,
        # Use fp16 for Colab (more stable than bf16 on T4)
        bf16=False,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit" if quant_config else "adamw_torch",
        report_to="wandb" if use_wandb else "none",
        run_name=run_name,
        dataset_text_field="text",  # Will be created by formatting_func
        packing=False,
        # Colab-specific settings
        dataloader_pin_memory=False,  # Can cause issues on Colab
        dataloader_num_workers=0,  # Avoid multiprocessing issues
    )
    
    # Create trainer
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        formatting_func=formatting_func,
    )
    
    # Train
    logger.info("Starting training on Colab...")
    trainer.train()
    
    # Save final model
    logger.info(f"Saving model to {output_dir}")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    # Save training config
    config = {
        "model_name": model_args.model_name,
        "lora_r": lora_args.lora_r,
        "lora_alpha": lora_args.lora_alpha,
        "lora_dropout": lora_args.lora_dropout,
        "target_modules": target_modules,
        "num_epochs": num_epochs,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "max_seq_length": data_args.max_seq_length,
        "use_4bit": model_args.use_4bit,
        "use_8bit": model_args.use_8bit,
        "colab_optimized": True,
    }
    
    with open(os.path.join(output_dir, "training_config.json"), 'w') as f:
        json.dump(config, f, indent=2)
    
    logger.info("Training complete!")
    
    return trainer

## Training Configuration

Configure your training parameters here. The default settings are optimized for Google Colab's free tier.

In [ ]:
# Training configuration (modify as needed)
model_args = ModelArguments(
    model_name=INSTRUCT_MODEL_NAME,
    use_4bit=True,  # Use 4-bit quantization for memory efficiency
    use_8bit=False
)

lora_args = LoraArguments(
    lora_r=16,  # LoRA rank
    lora_alpha=32,  # LoRA alpha
    lora_dropout=0.05,  # LoRA dropout
    target_modules="q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"  # Target modules
)

data_args = DataArguments(
    train_data="data/training/music_qa_train.json",
    val_data="data/training/music_qa_val.json",
    max_seq_length=1024  # Maximum sequence length
)

# Training hyperparameters (optimized for Colab)
training_config = {
    "output_dir": "models/qwen3-music-lora",
    "num_epochs": 2,  # Reduced epochs for faster training
    "batch_size": 1,  # Small batch size for Colab memory
    "gradient_accumulation_steps": 8,  # Effective batch size = 1 * 8 = 8
    "learning_rate": 2e-4,
    "warmup_ratio": 0.1,
    "logging_steps": 5,  # Log every 5 steps
    "save_steps": 50,  # Save checkpoint every 50 steps
    "eval_steps": 50,  # Evaluate every 50 steps
    "use_wandb": False,  # Set to True if you want W&B logging
    "run_name": "qwen3-music-lora-colab"
}

print("Training configuration:")
for key, value in training_config.items():
    print(f"{key}: {value}")
print("\nModel configuration:")
print(f"Model: {model_args.model_name}")
print(f"4-bit quantization: {model_args.use_4bit}")
print(f"LoRA rank: {lora_args.lora_r}")

## Mount Google Drive (Optional but Recommended)

Mount Google Drive to save your trained model and checkpoints persistently.

In [ ]:
# Mount Google Drive to save models persistently
from google.colab import drive
drive.mount('/content/drive')

# Update output directory to save to Drive
training_config["output_dir"] = "/content/drive/MyDrive/qwen3-music-lora"

print(f"Model will be saved to: {training_config['output_dir']}")

## Start Training

**⚠️ Important Notes:**
- Training may take 1-2 hours depending on your dataset size
- Colab may disconnect after 12 hours on free tier
- Use the save steps to resume training if disconnected
- Monitor GPU memory usage in the cell output

Click the play button below to start training!

In [ ]:
# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("❌ No GPU available. Please enable GPU runtime.")
    print("Go to Runtime → Change runtime type → Hardware accelerator → GPU")

# Start training
trainer = train(
    model_args=model_args,
    lora_args=lora_args,
    data_args=data_args,
    **training_config
)

print("\n🎉 Training completed successfully!")
print(f"Model saved to: {training_config['output_dir']}")

## Test the Trained Model

After training, you can test your fine-tuned model with some example prompts.

In [ ]:
# Load and test the trained model - FIXED version
from peft import PeftModel

# Load the fine-tuned model
model_path = training_config["output_dir"]
base_model = AutoModelForCausalLM.from_pretrained(
    model_args.model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Load LoRA weights
model = PeftModel.from_pretrained(base_model, model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

def generate_response(messages, max_new_tokens=256):
    """Generate response using proper method - extract only new tokens."""
    # Format using tokenizer's native chat template (same as training)
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    # Tokenize and track input length
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    input_length = inputs['input_ids'].shape[1]
    
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Extract ONLY the newly generated tokens
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    # Remove Qwen3 thinking tags <think>...</think>
    import re
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    
    return response.strip()

# Test prompts
test_prompts = [
    "What are some popular Vietnamese music genres?",
    "Tell me about the artist Sơn Tùng M-TP.",
    "What instruments are commonly used in traditional Vietnamese music?"
]

print("Testing the fine-tuned model:\n")
for prompt in test_prompts:
    print(f"Prompt: {prompt}")
    
    # Format as chat template
    messages = [{"role": "user", "content": prompt}]
    
    # Generate response using proper method
    assistant_response = generate_response(messages)
    
    print(f"Response: {assistant_response}")
    print("-" * 50)

## Model Evaluation

Evaluate your trained model on the test dataset.

In [ ]:
# Model Evaluation Cell - FIXED: Proper response extraction
import json
from tqdm import tqdm
import torch
from peft import PeftModel

def evaluate_music_model(model_path, test_data_path, output_path="data/evaluation/evaluation_results.json"):
    """Evaluate the fine-tuned music model."""
    
    print("🔍 Loading model for evaluation...")
    
    # Load model
    base_model = AutoModelForCausalLM.from_pretrained(
        model_args.model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base_model, model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    
    # Ensure pad token is set
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    # Load test data
    try:
        with open(test_data_path, 'r', encoding='utf-8') as f:
            test_data = json.load(f)
        print(f"📊 Loaded {len(test_data)} test samples")
    except FileNotFoundError:
        print("❌ Test data not found. Please upload music_qa_test.json")
        print("💡 You can create test data by splitting val data or using part of train data")
        # Fallback: use val data as test
        test_data_path = data_args.val_data
        try:
            with open(test_data_path, 'r', encoding='utf-8') as f:
                test_data = json.load(f)[:50]  # Use first 50 samples
            print(f"📊 Using validation data as test ({len(test_data)} samples)")
        except:
            print("❌ No test data available. Please check your data files.")
            return None
    
    def generate_answer(messages):
        """Generate answer from messages - FIXED version."""
        # Use the same chat template as training
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
        # Tokenize and get input length for extracting only new tokens
        inputs = tokenizer(formatted_prompt, return_tensors="pt")
        input_length = inputs['input_ids'].shape[1]
        
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        # Generate with proper settings
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,  # Greedy decoding for evaluation
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        # Extract ONLY the newly generated tokens (not the input prompt)
        new_tokens = outputs[0][input_length:]
        
        # Decode only the new tokens
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        # Clean up answer
        answer = answer.strip()
        
        # Remove Qwen3 thinking tags <think>...</think>
        import re
        answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
        
        # Remove any trailing special markers that might have been missed
        for marker in ['<|endoftext|>', '<|im_end|>', '</s>', '<|end|>']:
            answer = answer.replace(marker, '').strip()
        
        return answer
    
    # Evaluate
    results = []
    correct = 0
    total = min(50, len(test_data))  # Test on 50 samples max for speed
    
    print(f"⚡ Evaluating {total} samples...")
    
    for i, item in enumerate(tqdm(test_data[:total])):
        messages = item['messages']
        
        # Find expected answer
        expected_answer = ""
        user_question = ""
        
        for msg in messages:
            if msg['role'] == 'assistant':
                expected_answer = msg['content']
            elif msg['role'] == 'user':
                user_question = msg['content']
        
        # Generate prediction
        user_messages = [msg for msg in messages if msg['role'] != 'assistant']
        predicted_answer = generate_answer(user_messages)
        
        # Improved accuracy check
        import re
        
        def normalize_text(text):
            """Normalize text for comparison."""
            # Remove thinking tags if any remain
            text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
            # Normalize whitespace
            text = ' '.join(text.split())
            return text.lower().strip()
        
        pred_clean = normalize_text(predicted_answer)
        truth_clean = normalize_text(expected_answer)
        
        # Exact match
        is_correct = pred_clean == truth_clean
        
        # Fuzzy match: check if prediction contains the expected answer or vice versa
        if not is_correct:
            is_correct = truth_clean in pred_clean or pred_clean in truth_clean
        
        # Fuzzy match for yes/no questions
        if not is_correct:
            pred_yn = pred_clean.startswith(('yes', 'no'))
            truth_yn = truth_clean.startswith(('yes', 'no'))
            if pred_yn and truth_yn:
                pred_word = pred_clean.split()[0]
                truth_word = truth_clean.split()[0]
                is_correct = pred_word == truth_word
        
        if is_correct:
            correct += 1
        
        results.append({
            'question': user_question,
            'expected': expected_answer,
            'predicted': predicted_answer,
            'correct': is_correct
        })
    
    # Calculate accuracy
    accuracy = correct / total if total > 0 else 0
    
    print(f"\n🎯 Evaluation Results: {accuracy:.2%}")
    print(f"📈 Correct answers: {correct}/{total}")
    
    # Create evaluation directory
    os.makedirs('data/evaluation', exist_ok=True)
    
    # Save results
    eval_results = {
        'accuracy': accuracy,
        'total_samples': total,
        'correct_answers': correct,
        'results': results
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(eval_results, f, indent=2, ensure_ascii=False)
    
    print(f"💾 Results saved to {output_path}")
    
    # Show sample results
    print("\n📝 Sample Results:")
    for i, result in enumerate(results[:3]):
        status = "✅" if result['correct'] else "❌"
        print(f"{status} Q: {result['question'][:50]}...")
        print(f"   Expected: {result['expected']}")
        print(f"   Predicted: {result['predicted']}")
        print()
    
    return accuracy

# Run evaluation
print("=== MODEL EVALUATION ===")
eval_accuracy = evaluate_music_model(
    model_path=training_config["output_dir"],
    test_data_path="data/training/music_qa_test.json",
    output_path="data/evaluation/evaluation_results.json"
)

if eval_accuracy is not None:
    print(f"🎯 Overall Accuracy: {eval_accuracy:.2%}")
else:
    print("❌ Evaluation failed")

## Benchmark Comparison

Compare your model performance with baseline models.

In [ ]:
# Benchmark Comparison Cell
from datetime import datetime

def run_benchmark_comparison(lora_model_path, test_data_path, output_path="data/evaluation/benchmark_results.json"):
    """Compare fine-tuned model with baseline."""
    
    print("🔥 Running Benchmark Comparison...")
    
    # Load evaluation results
    try:
        with open("data/evaluation/evaluation_results.json", 'r', encoding='utf-8') as f:
            eval_results = json.load(f)
        lora_accuracy = eval_results['accuracy']
        print(f"LoRA Accuracy from previous evaluation: {lora_accuracy:.2%}")
    except FileNotFoundError:
        print("❌ Evaluation results not found. Run evaluation first!")
        return None
    
    # Baseline accuracies (estimated from literature/training data)
    baselines = {
        "Qwen3-0.6B Baseline": 0.48,  # No fine-tuning
        "GPT-3.5-turbo": 0.47,        # No graph context
        "GPT-4o-mini": 0.56,          # No graph context
        "Qwen3-0.6B + LoRA (Ours)": lora_accuracy
    }
    
    print("\n📊 BENCHMARK COMPARISON")
    print("=" * 50)
    
    best_model = "Qwen3-0.6B + LoRA (Ours)"
    max_acc = lora_accuracy
    
    for model, acc in baselines.items():
        marker = "🏆" if model == best_model else "  "
        print(f"{marker} {model}: {acc:.2%}")
        
        if acc > max_acc:
            max_acc = acc
            best_model = model
    
    print("=" * 50)
    
    # Performance analysis
    improvement = lora_accuracy - baselines["Qwen3-0.6B Baseline"]
    print(f"\n📈 Performance Analysis: {improvement:+.2%}")
    
    if improvement > 0:
        print("🎉 Your model performs better than baseline!")
    else:
        print("🤔 Model needs more training or better data")
        print("💡 Try increasing epochs or checking training data quality")
    
    # Question type analysis (if available in test data)
    try:
        with open(test_data_path, 'r', encoding='utf-8') as f:
            test_data = json.load(f)
        
        question_types = {}
        for item in test_data[:100]:  # Sample analysis
            if 'metadata' in item and 'question_type' in item['metadata']:
                qtype = item['metadata']['question_type']
                if qtype not in question_types:
                    question_types[qtype] = 0
                question_types[qtype] += 1
        
        if question_types:
            print("\n🎯 Question Types in Dataset:")
            for qtype, count in question_types.items():
                print(f"  {qtype}: {count} questions")
    except:
        pass
    
    # Save benchmark results
    os.makedirs('data/evaluation', exist_ok=True)
    benchmark_data = {
        'timestamp': str(datetime.now()),
        'models': baselines,
        'best_model': best_model,
        'improvement_over_baseline': improvement,
        'our_accuracy': lora_accuracy
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(benchmark_data, f, indent=2, ensure_ascii=False)
    
    print(f"\n💾 Benchmark results saved to {output_path}")
    
    return baselines

# Run benchmark
print("=== BENCHMARK COMPARISON ===")
benchmark_results = run_benchmark_comparison(
    lora_model_path=training_config["output_dir"],
    test_data_path="data/training/music_qa_test.json",
    output_path="data/evaluation/benchmark_results.json"
)

if benchmark_results:
    print("\n✅ Benchmark completed!")
    print("🏆 Your model is ready for deployment!")
else:
    print("\n❌ Benchmark failed - run evaluation first")

## Download Model (Optional)

If you didn't mount Google Drive, you can download the trained model to your local machine.

In [ ]:
# Zip and download the model (if not using Drive)
import shutil
from google.colab import files

model_path = training_config["output_dir"]
zip_path = "/content/qwen3-music-lora.zip"

# Zip the model directory
shutil.make_archive("/content/qwen3-music-lora", 'zip', model_path)

# Download
files.download(zip_path)

print("Model downloaded as qwen3-music-lora.zip")